In [1]:
from utils import prepare_dataframe, merge_csv_files_in_directory, filterForEqualLabelCount,filterRepoName

embedding_directory = 'embeddings/GCB_Pooler_embeddings_content_before_only_apache_balanced_46008'
model_directory = 'models/GCB_Pooler_embeddings_content_before_only_apache_balanced_46008.pth'
column = "content_before"
columns_to_keep = [ column]

#merge_csv_files_in_directory()
df = prepare_dataframe("merged.csv", shouldBalance=False, should_merge_csv=False, columns_to_keep=columns_to_keep) 
df = filterRepoName(df,"apache")
df = filterForEqualLabelCount(df=df)
resulting_label_count = df['labels'].nunique()
df


labels
enhancement    37586
bug            32864
clean          17263
performance     5392
refactor        4596
Name: count, dtype: int64
labels
enhancement    37586
bug            32864
clean          17263
Name: count, dtype: int64
15336


,full_repo_name,labels,content_before,index
90648,apache/rocketmq,enhancement,/*\n * Licensed to the Apache Software Foundat...,90648
85016,apache/pinot,enhancement,/**\n * Licensed to the Apache Software Founda...,85016
52956,apache/maven-dependency-plugin,enhancement,/*\n * Licensed to the Apache Software Foundat...,52956
43844,apache/cloudstack,enhancement,// Licensed to the Apache Software Foundation ...,43844
93757,apache/accumulo,enhancement,/*\n * Licensed to the Apache Software Foundat...,93757
...,...,...,...,...
63863,apache/pulsar,clean,/*\n * Licensed to the Apache Software Foundat...,63863
71156,apache/netbeans,clean,/*\n * Licensed to the Apache Software Foundat...,71156
78185,apache/netbeans,clean,/*\n * Licensed to the Apache Software Foundat...,78185
42537,apache/cloudstack,clean,// Licensed to the Apache Software Foundation ...,42537


In [2]:
import pandas as pd
import torch


# One-hot encoding ile labels sütununu dönüştürme
df_encoded_labels = pd.get_dummies(df, columns=['labels'])
df_encoded_repo = pd.get_dummies(df, columns=['full_repo_name'])

# Özellikler ve etiketler
X_repo = df_encoded_repo.drop(columns=['labels',column,'index'])
X_repo = X_repo.astype(int)

# y = df_encoded.drop(columns=['content_before', 'content_after', 'tokens'])
y = df_encoded_labels.drop(columns=[column, 'full_repo_name','index'])123

y = y.astype(int) # convert it 0,1     instead of true, false 

df['label_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in y.values] 
df['repo_name_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in X_repo.values] 

df = df.drop(columns=['full_repo_name', 'labels',column])


#print(df['label_tensor'][0] )
#print(df['repo_name_tensor'][0] )
df

,index,label_tensor,repo_name_tensor
90648,90648,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
85016,85016,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
52956,52956,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
43844,43844,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
93757,93757,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(1.), tensor(0.), tensor(0.), tensor(0...."
...,...,...,...
63863,63863,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
71156,71156,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
78185,78185,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
42537,42537,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."


In [3]:
from utils import load_embeddings_from_directory_for_df
import pandas as pd

df = load_embeddings_from_directory_for_df(directory_path=embedding_directory,df=df)


  0%|          | 0/46008 [00:00<?, ?it/s]

In [4]:
from sklearn.model_selection import train_test_split

X_index = df[['index']]  
# Split the code data into test, train and validation sets
X_index_train, X_index_test = train_test_split(X_index, test_size=0.3, random_state=42) #random_state: 0,3074,42
X_index_test, X_index_val = train_test_split(X_index_test, test_size=0.5, random_state=42)

In [5]:
import gc
import torch

df_train = df[df['index'].isin(X_index_train['index'].values)].reset_index(drop=True)
df_val = df[df['index'].isin(X_index_val['index'].values)].reset_index(drop=True)
df_test = df[df['index'].isin(X_index_test['index'].values)].reset_index(drop=True)

torch.cuda.empty_cache()

df_train

,index,label_tensor,repo_name_tensor,embedding
0,52956,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.2708), tensor(-0.0817), tensor(-0.2..."
1,43844,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.3480), tensor(-0.0658), tensor(-0.1..."
2,90569,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.3614), tensor(-0.0802), tensor(-0.0..."
3,59460,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.3303), tensor(-0.0807), tensor(-0.1..."
4,91168,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.3223), tensor(-0.0390), tensor(-0.1..."
...,...,...,...,...
32200,76968,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.1110), tensor(-0.1491), tensor(-0.1..."
32201,63863,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.2979), tensor(-0.1422), tensor(-0.0..."
32202,78185,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.2940), tensor(-0.0417), tensor(-0.0..."
32203,42537,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0....","[tensor(-0.3618), tensor(-0.0180), tensor(-0.2..."


In [6]:
from torch.utils.data import DataLoader, Dataset
from utils import load_embedding_from_directory

# Custom Dataset Class to handle the dataframe
class CodeDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.id_list = df['index'].values 

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
    
        index = self.id_list[idx]  

        tensor_3d = self.df.loc[self.df['index'] == index, 'embedding'].iloc[0]
        repo_name_tensor = self.df.loc[self.df['index'] == index, 'repo_name_tensor'].iloc[0]
        label_tensor = self.df.loc[self.df['index'] == index, 'label_tensor'].iloc[0]
        
        #print(f"{repo_name_tensor.shape} {tensor_3d.shape} {label_tensor.shape}")
        #print("-------------------------------")
        
        #Flattening
        #embedding = torch.flatten(tensor_3d)
        embedding = tensor_3d
        return embedding, label_tensor, repo_name_tensor, index


In [7]:
# Create DataLoader

# Creating a DataLoader
def create_dataloader(df, batch_size=32, shuffle=True):
    dataset = CodeDataset(df)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

batch_size = 128
test_loader = create_dataloader(df=df_test, batch_size=batch_size)
train_loader = create_dataloader(df=df_train, batch_size=batch_size)
val_loader = create_dataloader(df=df_val,batch_size=batch_size)

full_loader_for_scaler = create_dataloader(df=df, batch_size=512)

In [8]:
from sklearn.preprocessing import StandardScaler
import tqdm

scaler = StandardScaler()

for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(full_loader_for_scaler):
    scaler.partial_fit(batch_X)

/tmp/ipykernel_3408137/796712526.py:6: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(full_loader_for_scaler):


  0%|          | 0/90 [00:00<?, ?it/s]

In [9]:
import joblib

# Save scaler into file
joblib.dump(scaler, 'scalers/scaler_pooler_3labels_15336.pkl')

['scalers/scaler_pooler_3labels_15336.pkl']

In [10]:
import joblib

# Load scaler from file
scaler = joblib.load('scalers/scaler_pooler_3labels_15336.pkl')

In [11]:
import torch.nn as nn

# Define the ANN model
class CodeDiffNN(nn.Module):
    def __init__(self, code_input_size, repo_input_size, hidden_sizes_embedding_pipeline,hidden_sizes_reponame_pipeline,hidden_sizes_merged_pipeline, output_size, dropout_rate):
        super(CodeDiffNN, self).__init__()
        
        # Pipeline for code embeddings
        self.code_fc1 = nn.Linear(code_input_size, hidden_sizes_embedding_pipeline[0])
        self.code_bn1 = nn.BatchNorm1d(hidden_sizes_embedding_pipeline[0])
        self.code_relu1 = nn.ReLU()
        self.code_dropout1 = nn.Dropout(dropout_rate)
        
        self.code_fc2 = nn.Linear(hidden_sizes_embedding_pipeline[0], hidden_sizes_embedding_pipeline[1])
        self.code_bn2 = nn.BatchNorm1d(hidden_sizes_embedding_pipeline[1])
        self.code_relu2 = nn.ReLU()
        self.code_dropout2 = nn.Dropout(dropout_rate)
        
        self.code_fc3 = nn.Linear(hidden_sizes_embedding_pipeline[1], hidden_sizes_embedding_pipeline[2])
        self.code_bn3 = nn.BatchNorm1d(hidden_sizes_embedding_pipeline[2])
        self.code_relu3 = nn.ReLU()
        self.code_dropout3 = nn.Dropout(dropout_rate)  
        
        
        # Pipeline for repo names
        self.repo_fc1 = nn.Linear(repo_input_size, hidden_sizes_reponame_pipeline[0])
        self.repo_bn1 = nn.BatchNorm1d(hidden_sizes_reponame_pipeline[0])
        self.repo_relu1 = nn.ReLU()
        self.repo_dropout1 = nn.Dropout(dropout_rate)
        
        self.repo_fc2 = nn.Linear(hidden_sizes_reponame_pipeline[0], hidden_sizes_reponame_pipeline[1])
        self.repo_bn2 = nn.BatchNorm1d(hidden_sizes_reponame_pipeline[1])
        self.repo_relu2 = nn.ReLU()
        self.repo_dropout2 = nn.Dropout(dropout_rate)

        
        # Combining the outputs of both pipelines
        combined_input_size = hidden_sizes_embedding_pipeline[2] + hidden_sizes_reponame_pipeline[1]   # Since we're combining two pipelines
        self.combined_fc1 = nn.Linear(combined_input_size, hidden_sizes_merged_pipeline[0])
        self.combined_bn1 = nn.BatchNorm1d(hidden_sizes_merged_pipeline[0])
        self.combined_relu1 = nn.ReLU()
        self.combined_dropout1 = nn.Dropout(dropout_rate)

        self.combined_fc2  = nn.Linear(hidden_sizes_merged_pipeline[0], hidden_sizes_merged_pipeline[1])
        self.combined_bn2 = nn.BatchNorm1d(hidden_sizes_merged_pipeline[1])
        self.combined_relu2  = nn.ReLU()
        self.combined_dropout2 = nn.Dropout(dropout_rate)
        
        self.combined_fc3  = nn.Linear(hidden_sizes_merged_pipeline[1], hidden_sizes_merged_pipeline[2])
        self.combined_bn3 = nn.BatchNorm1d(hidden_sizes_merged_pipeline[2])
        self.combined_relu3  = nn.ReLU()
        self.combined_dropout3 = nn.Dropout(dropout_rate)
        
        self.combined_fc4  = nn.Linear(hidden_sizes_merged_pipeline[2], hidden_sizes_merged_pipeline[3])
        self.combined_bn4 = nn.BatchNorm1d(hidden_sizes_merged_pipeline[3])
        self.combined_relu4  = nn.ReLU()
        self.combined_dropout4 = nn.Dropout(dropout_rate)

        # Output layer
        self.fc_out = nn.Linear(hidden_sizes_merged_pipeline[3], output_size)
        
    def forward(self, code_x, repo_x):
        # Process code embeddings
        code_x = self.code_fc1(code_x)
        code_x = self.code_bn1(code_x)
        code_x = self.code_relu1(code_x)
        code_x = self.code_dropout1(code_x)
        
        code_x = self.code_fc2(code_x)
        code_x = self.code_bn2(code_x)
        code_x = self.code_relu2(code_x)
        code_x = self.code_dropout2(code_x)
        
        code_x = self.code_fc3(code_x)
        code_x = self.code_bn3(code_x)
        code_x = self.code_relu3(code_x)
        code_x = self.code_dropout3(code_x)
        
        # Process repo names
        repo_x = self.repo_fc1(repo_x)
        repo_x = self.repo_bn1(repo_x)
        repo_x = self.repo_relu1(repo_x)
        repo_x = self.repo_dropout1(repo_x)
        
        repo_x = self.repo_fc2(repo_x)
        repo_x = self.repo_bn2(repo_x)
        repo_x = self.repo_relu2(repo_x)
        repo_x = self.repo_dropout2(repo_x)
        
        # Combine both outputs
        combined_x = torch.cat((code_x, repo_x), dim=1)
        
        # Process the combined output
        combined_x = self.combined_fc1(combined_x)
        combined_x = self.combined_bn1(combined_x)
        combined_x = self.combined_relu1(combined_x)
        combined_x = self.combined_dropout1(combined_x)

        combined_x = self.combined_fc2(combined_x)
        combined_x = self.combined_bn2(combined_x)
        combined_x = self.combined_relu2(combined_x)
        combined_x = self.combined_dropout2(combined_x)
        
        combined_x = self.combined_fc3(combined_x)
        combined_x = self.combined_bn3(combined_x)
        combined_x = self.combined_relu3(combined_x)
        combined_x = self.combined_dropout3(combined_x)
        
        combined_x = self.combined_fc4(combined_x)
        combined_x = self.combined_bn4(combined_x)
        combined_x = self.combined_relu4(combined_x)
        combined_x = self.combined_dropout4(combined_x)
        
        # Final output
        output = self.fc_out(combined_x)
        return output


In [12]:
import torch.optim as optim
# Define model parameters

code_embedding_dim = df_train['embedding'][5].shape[0]

repo_input_size = len(df_train['repo_name_tensor'][0])
#hidden_sizes = [1024, 512, 256, 128]
hidden_sizes_embedding_pipeline = [1024, 2048, 4096]
hidden_sizes_reponame_pipeline = [512, 1024]
hidden_sizes_merged_pipeline = [2048, 1024, 512, 256]
output_size = resulting_label_count
dropout_rate = 0.05

device = "cuda:0"
model = CodeDiffNN(code_embedding_dim, repo_input_size, hidden_sizes_embedding_pipeline,hidden_sizes_reponame_pipeline,hidden_sizes_merged_pipeline, output_size, dropout_rate).to(device)
# model = SinglePipelineCodeDiffNN(code_embedding_dim, hidden_sizes, output_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

# Decrease learning rate automatically on plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=400, factor=0.1, verbose=True)

In [13]:
print(resulting_label_count)
print(code_embedding_dim)
torch.cuda.empty_cache()

3
768


In [14]:
# Function to calculate accuracy
def calculate_accuracy(outputs, labels):
    _, predicted = torch.max(outputs, 1)  # Get the index of the max log-probability
    # labels_1d = torch.argmax(labels, dim=1) # turn into 1D tensor
    correct = (predicted == labels).sum().item()
    accuracy = correct / labels.size(0)  # Accuracy = number of correct predictions / total number of samples
    return accuracy

In [15]:
import tqdm

def train_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(loader):
        
        batch_X = scaler.transform(batch_X)
        batch_X = torch.tensor(batch_X,dtype=torch.float32)
        
        # Forward pass        
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        batch_y = torch.argmax(batch_y, dim=1)

        # outputs = model(batch_X)
        outputs = model(batch_X, batch_X_repo)
        loss = criterion(outputs, batch_y)
    
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy
        
        
    val_loss = running_loss / len(loader)
    val_accuracy = running_accuracy / len(loader)
    
    return val_loss, val_accuracy

In [16]:

import time
from sklearn.metrics import precision_score, recall_score, f1_score


def evaluate_with_data_loader(model, loader):
    
    device = "cpu"
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    all_labels = []
    all_predictions = []
    
    start_time = time.time()  # Inference zamanını başlat
    
    for batch_X, batch_y, batch_X_repo, idx in loader:
        batch_X = scaler.transform(batch_X)
        batch_X = torch.tensor(batch_X, dtype=torch.float32).to(device)  # batch_X'i cihaza taşı
        
        # Forward pass
        batch_y = batch_y.to(device)  # batch_y'yi cihaza taşı
        batch_X_repo = batch_X_repo.to(device)  # batch_X_repo'yu cihaza taşı
        
        batch_y = torch.argmax(batch_y, dim=1)

        outputs = model(batch_X, batch_X_repo)  # Model cihaza zaten taşınmış olmalı
        loss = criterion(outputs, batch_y)
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y)
        
        # Store labels and predictions for precision, recall, F1-score
        _, predicted = torch.max(outputs, 1)
        all_labels.extend(batch_y.cpu().numpy())  # CPU'ya taşı ve numpy olarak ekle
        all_predictions.extend(predicted.cpu().numpy())  # CPU'ya taşı ve numpy olarak ekle
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy
        
    end_time = time.time()  # Inference zamanını bitir
    inference_time = end_time - start_time  # Toplam inference zamanı
    
    val_loss = running_loss / len(loader)
    val_accuracy = running_accuracy / len(loader)
    
    # Calculate precision, recall, and F1-score
    precision = precision_score(all_labels, all_predictions, average='weighted')
    recall = recall_score(all_labels, all_predictions, average='weighted')
    f1 = f1_score(all_labels, all_predictions, average='weighted')
    
    # Adjust learning rate based on validation loss
    scheduler.step(val_loss)
    device = "cuda:0"
    return val_loss, val_accuracy, precision, recall, f1, inference_time



In [17]:
def test_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    predicted_y = []
    y = []
    
    for batch_X, batch_y, batch_X_repo, idx in loader:
        
        batch_X = scaler.transform(batch_X)
        batch_X = torch.tensor(batch_X,dtype=torch.float32)

        # Forward pass
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        # index encoding
        batch_y_1D = torch.argmax(batch_y, dim=1)

        outputs = model(batch_X, batch_X_repo)
        # outputs = model(batch_X)
        loss = criterion(outputs, batch_y_1D)
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y_1D)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy

        predicted_y.append(outputs)
        y.append(batch_y)
        
    test_loss = running_loss / len(loader)
    test_accuracy = running_accuracy / len(loader)
    
    return test_loss, test_accuracy, torch.cat(predicted_y, dim=0), torch.cat(y, dim=0)

In [18]:
import copy
import tqdm

# Train the model

# Initialize variables to track the best model and best accuracy
best_accuracy = 0.0

num_epochs = 100

for epoch in tqdm.tqdm_notebook(range(num_epochs)):
    model.train()  # Set the model to training mode
    model.to("cuda:0")  # Modeli cihaza taşı
    
    # Print average loss and accuracy for the epoch
    avg_loss,avg_accuracy = train_with_data_loader(model= model,loader= train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}')

    # Evaluate on validation data
    model.eval()  # Set the model to evaluation mode
    model.to("cpu")  # Modeli cihaza taşı
    
    with torch.no_grad():  # Disable gradient computation for validation
        val_loss, val_accuracy, precision, recall, f1, inference_time = evaluate_with_data_loader(model, val_loader)
        print(f"Inference Time: {inference_time:.4f} seconds")
        print(f"Loss:{val_loss:.4f}, Accuracy: {val_accuracy:.4f}")
        print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")
    # Check if the current model has the best validation accuracy
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        torch.save(model.state_dict(), model_directory)
    



/tmp/ipykernel_3408137/3597344509.py:11: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for epoch in tqdm.tqdm_notebook(range(num_epochs)):


  0%|          | 0/100 [00:00<?, ?it/s]

/tmp/ipykernel_3408137/2221030542.py:8: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(loader):


  0%|          | 0/252 [00:00<?, ?it/s]

Epoch [1/100], Loss: 0.6374, Accuracy: 0.7070
Inference Time: 6.2903 seconds
Loss:0.5335, Accuracy: 0.7516
Precision: 0.7558, Recall: 0.7517, F1-Score: 0.7473


...

  0%|          | 0/252 [00:00<?, ?it/s]

Epoch [99/100], Loss: 0.1139, Accuracy: 0.9458
Inference Time: 6.2761 seconds
Loss:1.2976, Accuracy: 0.8248
Precision: 0.8253, Recall: 0.8248, F1-Score: 0.8250


/tmp/ipykernel_3408137/2221030542.py:8: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(loader):


  0%|          | 0/252 [00:00<?, ?it/s]

Epoch [100/100], Loss: 0.1131, Accuracy: 0.9452
Inference Time: 6.4564 seconds
Loss:1.3349, Accuracy: 0.8235
Precision: 0.8247, Recall: 0.8235, F1-Score: 0.8240


In [19]:
# Load model from file
model = CodeDiffNN(code_embedding_dim, repo_input_size, hidden_sizes_embedding_pipeline,hidden_sizes_reponame_pipeline, hidden_sizes_merged_pipeline, output_size, dropout_rate).to(device)
model.load_state_dict(torch.load(model_directory))

<All keys matched successfully>

In [20]:
with torch.no_grad():
    # Load best model from previous training cycle

    model.eval()
    model.to("cpu")
    
    val_loss, val_accuracy, precision, recall, f1, inference_time = evaluate_with_data_loader(model, test_loader)
    print(f'Accuracy on test data: {val_accuracy}')
    print(f"Inference Time: {inference_time} seconds")
    print(f"Loss:{val_loss}")
    print(f"Precision: {precision}, Recall: {recall}, F1-Score: {f1}")

Accuracy on test data: 0.815727386039886
Inference Time: 6.213409662246704 seconds
Loss:0.9830954880626114
Precision: 0.8175245627832689, Recall: 0.815678887117809, F1-Score: 0.8164334492125461


In [21]:
with torch.no_grad():
    # Load best model from previous training cycle
    model.to("cuda:0")
    model.eval()
    test_loss,test_accuracy,test_outputs,y_test = test_with_data_loader(model=model,loader=test_loader)
    
    print(f'Accuracy on test data: {test_accuracy:.4f}')
    print(test_outputs.shape)

Accuracy on test data: 0.8156
torch.Size([6901, 3])


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

_, predicted = torch.max(test_outputs.detach().cpu(), 1)  # Get the index of the max log-probability
labels_1d = torch.argmax(y_test, dim=1).cpu() # turn into 1D tensor

# Convert to numpy arrays
y_test_np = labels_1d.numpy()
predicted_np = predicted.numpy()

# Compute confusion matrix
cm = confusion_matrix(y_test_np, predicted_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('Confusion Matrix for Multi-Class Classification')
plt.show()

print(predicted_np)
print(y_test_np)

In [ ]:
import gc
# Free up GPU memory (testing)
print(f"Cached GPU memory before: {torch.cuda.memory_reserved()} bytes")
del model
del test_outputs
del y_test
del predicted
del labels_1d
del val_loader, test_loader, train_loader
del cm
gc.collect()
torch.cuda.empty_cache()
print(f"Cached GPU memory after: {torch.cuda.memory_reserved()} bytes")

In [ ]:
print(df.shape)
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

In [ ]:
df_ml_train = pd.concat([df_train, df_val], axis=0, ignore_index=True)
df_ml_train

In [ ]:
import numpy as np
# Özellikleri birleştirme ve normalize etme
X_train = np.vstack(df_train.apply(lambda row: np.concatenate([row['embedding'], row['repo_name_tensor']]), axis=1))

# Label tensor'leri argmax ile sınıf etiketine çevirme
y_train = df_train['label_tensor'].apply(lambda x: x.numpy().argmax() if isinstance(x, torch.Tensor) else x).values

X_test = np.vstack(df_test.apply(lambda row: np.concatenate([row['embedding'], row['repo_name_tensor']]), axis=1))

y_test = df_test['label_tensor'].apply(lambda x: x.numpy().argmax() if isinstance(x, torch.Tensor) else x).values

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
from utils import prepare_dataframe, load_embeddings_from_directory_for_df
import time
import csv
import pandas as pd
import torch
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Modelleri tanımlıyoruz
models = [
    ("XGBoost", xgb.XGBClassifier(objective="multi:softmax", num_class=3, eval_metric="mlogloss")),
    ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("CatBoost", CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, verbose=0)),
    ("LightGBM", LGBMClassifier(num_class=3)),
    ("KNN", KNeighborsClassifier(n_neighbors=5)),
    ("NaiveBayes", GaussianNB()),
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42)),
    ("AdaBoost", AdaBoostClassifier(n_estimators=100, random_state=42)),
    ("SVM", SVC(decision_function_shape='ovo', kernel='linear', random_state=42))
]
# Modelleri eğitip tahmin sürelerini hesapla ve hemen CSV'ye kaydet
output_file = "model_metrics.csv"

# CSV dosyasını başlatma
with open(output_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # Başlıkları yaz
    writer.writerow(["Model Name", "Accuracy", "Precision", "Recall", "F1 Score", "Prediction Time (s)"])

metrics ={} 
# Modelleri işlemeye başla
for model_name, model in models:
    # Modeli eğit
    model.fit(X_train, y_train)

    # Tahmin süresini ölç
    start_time = time.time()
    y_pred = model.predict(X_test)  
    end_time = time.time()
    prediction_time = end_time - start_time  

    # Metrikleri hesapla
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    
    metrics[model_name] = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Prediction Time (s)": prediction_time
    }

    # Metrikleri ve tahmin süresini yazdır
    print(f"Metrics for {model_name}:")
    print(f"  Accuracy: {accuracy}")
    print(f"  Precision: {precision}")
    print(f"  Recall: {recall}")
    print(f"  F1 Score: {f1}")
    print(f"  Prediction Time (s): {prediction_time:.4f}")

    # Metrikleri CSV'ye yaz
    with open(output_file, mode='a', newline='', encoding='utf-8') as file:  
        writer = csv.writer(file)
        writer.writerow([
            model_name,
            accuracy,
            precision,
            recall,
            f1,
            prediction_time
        ])

print(f"\nMetrics saved incrementally to {output_file}")


In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Eğitilmiş modeller ile confusion matrix'i hesaplama
confusion_matrices = {}
for model_name, model in models:
    y_pred = model.predict(X_test)  # Test verisi üzerinde tahmin yap
    cm = confusion_matrix(y_test, y_pred)
    confusion_matrices[model_name] = cm  # Confusion matrix'i sakla

# Confusion matrix'leri görselleştirme ve tek bir resimde kaydetme
num_models = len(confusion_matrices)
rows = (num_models + 2) // 3  # 3 sütun, gereken satır sayısını hesapla
fig, axes = plt.subplots(nrows=rows, ncols=3, figsize=(18, 6 * rows))

fig.suptitle("Confusion Matrices for Trained Models", fontsize=20)

# Tüm confusion matrix'leri çiz
for ax, (model_name, cm) in zip(axes.flatten(), confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(f"{model_name} Accuracy:{metrics[model_name]['Accuracy']}")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

# Boş kalan eksenleri kapat
for i in range(num_models, len(axes.flatten())):
    fig.delaxes(axes.flatten()[i])

# Layout'u ayarla ve resmi kaydet
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
output_file = "all_confusion_matrices.png"
plt.savefig(output_file)
plt.close()

print(f"All confusion matrices saved to {output_file}")
